# 9.9 — Reading and Writing Kafka

**Chapter 9, §9.5.1.** *What does a Kafka record look like once it reaches Spark, and
what makes a topic replayable?*

The scripts of 9.1 run the live pipeline: a producer on one side, a streaming query on
the other. This notebook does the opposite. It stops the topic still and reads it as an
ordinary bounded table, because almost everything the chapter claims about Kafka is
easier to see when nothing is moving.

Four things are shown here and nowhere else in the course:

1. the exact row Spark gets from Kafka, which is the schema tabulated in the chapter;
2. why every Kafka query begins by casting and parsing, and what happens if you forget;
3. that a topic is a log, so reading it twice gives the same rows, and a range of offsets
   can be read back on demand;
4. that the streaming form of the same query differs by one method name.

**Before running this, start the broker:**

```bash
code/scripts/kafka/start.sh
```

Nothing else is needed. The notebook creates its own topic, writes to it, and cleans up
after itself, so it does not disturb the `weatherdata` topic the scripts use.

In [1]:
import json, logging, os, shutil, tempfile, time

from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import UnknownTopicOrPartitionError

BOOTSTRAP = "localhost:9092"
TOPIC = "cs777-notebook"
PARTITIONS = 3

# Fail early and say something useful, rather than letting Spark time out on a
# broker that is not there.
try:
    admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP, request_timeout_ms=5000)
except Exception as exc:
    raise RuntimeError(
        f"No Kafka broker at {BOOTSTRAP}. Start one with code/scripts/kafka/start.sh"
    ) from exc

# Start from an empty topic every time, so the offsets printed below begin at 0
# and a second run of the notebook prints the same numbers as the first.
try:
    admin.delete_topics([TOPIC])
except UnknownTopicOrPartitionError:
    pass
for _ in range(30):
    if TOPIC not in admin.list_topics():
        break
    time.sleep(1)
admin.create_topics([NewTopic(name=TOPIC, num_partitions=PARTITIONS, replication_factor=1)])
print(f"topic {TOPIC} created with {PARTITIONS} partitions")

topic cs777-notebook created with 3 partitions


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_json, struct
from pyspark.sql.types import DoubleType, StringType, StructField, StructType, TimestampType
from pyspark.errors import AnalysisException

SCRATCH = os.path.join(tempfile.gettempdir(), "cs777", "ch09-kafka")
shutil.rmtree(SCRATCH, ignore_errors=True)
os.makedirs(SCRATCH, exist_ok=True)

# The Kafka connector is not bundled with Spark. Requesting it here means this
# notebook needs no special launch command; the first run fetches it from Maven
# Central and later runs use the cache.
spark = (
    SparkSession.builder.appName("CS777-9.9")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", 4)
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# One cell below raises an analysis error on purpose. PySpark logs a structured
# record with the whole JVM stack trace for every one of those, so quiet it.
logging.getLogger("DataFrameQueryContextLogger").setLevel(logging.CRITICAL)
spark.version

:: loading settings :: url = jar:file:/Users/dt/.venvs/cs777-spark/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/dt/.ivy2.5.2/cache
The jars for the packages stored in: /Users/dt/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-90ceed63-cac2-48b0-87a7-66604a71edfe;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central

'4.2.0'

## Writing: a DataFrame becomes Kafka records

Kafka takes bytes, so a writer has to hand it a `key` and a `value` column and nothing
else is required. Six readings from four stations, with the station as the key and the
reading as a JSON string.

In [3]:
READINGS = [
    ("boston",   "2026-01-01 12:00:00", 12.1),
    ("denver",   "2026-01-01 12:00:01",  3.8),
    ("chicago",  "2026-01-01 12:00:02",  8.4),
    ("portland", "2026-01-01 12:00:03", 11.2),
    ("boston",   "2026-01-01 12:00:04", 12.6),
    ("denver",   "2026-01-01 12:00:05",  4.1),
]
readings = spark.createDataFrame(READINGS, "station STRING, reading_time STRING, temperature_c DOUBLE")

# key and value are the only columns Kafka cares about. The payload is packed
# into one JSON string, which is what almost every producer does in practice.
outgoing = readings.select(
    col("station").alias("key"),
    to_json(struct("station", "reading_time", "temperature_c")).alias("value"),
)
outgoing.show(truncate=False)

+--------+--------------------------------------------------------------------------------+
|key     |value                                                                           |
+--------+--------------------------------------------------------------------------------+
|boston  |{"station":"boston","reading_time":"2026-01-01 12:00:00","temperature_c":12.1}  |
|denver  |{"station":"denver","reading_time":"2026-01-01 12:00:01","temperature_c":3.8}   |
|chicago |{"station":"chicago","reading_time":"2026-01-01 12:00:02","temperature_c":8.4}  |
|portland|{"station":"portland","reading_time":"2026-01-01 12:00:03","temperature_c":11.2}|
|boston  |{"station":"boston","reading_time":"2026-01-01 12:00:04","temperature_c":12.6}  |
|denver  |{"station":"denver","reading_time":"2026-01-01 12:00:05","temperature_c":4.1}   |
+--------+--------------------------------------------------------------------------------+



In [4]:
(
    outgoing.write.format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("topic", TOPIC)
    .save()
)
print("written")

written


## Reading: the row Spark actually gets

This is the schema the chapter tabulates. The payload is the binary `key` and `value`
pair; everything else describes where the record came from. Note that `spark.read` is
used here, not `spark.readStream`: a topic has an end at any given moment, so it can be
read as a bounded table.

In [5]:
raw = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .load()
)
raw.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [6]:
# Truncated on purpose. key and value print as hex because they are bytes: Kafka
# stores the payload uninterpreted and has no idea what is in it.
(
    raw.select("key", "value", "topic", "partition", "offset")
    .orderBy("partition", "offset")
    .show(truncate=28)
)

+-------------------------+----------------------------+--------------+---------+------+
|                      key|                       value|         topic|partition|offset|
+-------------------------+----------------------------+--------------+---------+------+
|[70 6F 72 74 6C 61 6E 64]|[7B 22 73 74 61 74 69 6F ...|cs777-notebook|        0|     0|
|   [63 68 69 63 61 67 6F]|[7B 22 73 74 61 74 69 6F ...|cs777-notebook|        1|     0|
|      [64 65 6E 76 65 72]|[7B 22 73 74 61 74 69 6F ...|cs777-notebook|        2|     0|
|      [62 6F 73 74 6F 6E]|[7B 22 73 74 61 74 69 6F ...|cs777-notebook|        2|     1|
|      [64 65 6E 76 65 72]|[7B 22 73 74 61 74 69 6F ...|cs777-notebook|        2|     2|
|      [62 6F 73 74 6F 6E]|[7B 22 73 74 61 74 69 6F ...|cs777-notebook|        2|     3|
+-------------------------+----------------------------+--------------+---------+------+



## Casting and parsing

Two steps, and every Kafka query in the chapter begins with them. First cast the bytes
to strings, then parse the string against a schema the application declares. Kafka
cannot supply that schema; it never knew one.

Skipping the cast is not a subtle mistake. Spark checks the types before the query runs
and refuses it, which is the failure you want: loud, immediate, and naming the column.

In [7]:
PAYLOAD = StructType([
    StructField("station", StringType(), True),
    StructField("reading_time", TimestampType(), True),
    StructField("temperature_c", DoubleType(), True),
])

parsed = (
    raw.selectExpr("CAST(key AS STRING) AS station_key", "CAST(value AS STRING) AS payload",
                   "partition", "offset")
    .select("station_key", "partition", "offset", from_json(col("payload"), PAYLOAD).alias("r"))
    .select("station_key", "partition", "offset", "r.*")
)
parsed.orderBy("partition", "offset").show(truncate=False)

+-----------+---------+------+--------+-------------------+-------------+
|station_key|partition|offset|station |reading_time       |temperature_c|
+-----------+---------+------+--------+-------------------+-------------+
|portland   |0        |0     |portland|2026-01-01 12:00:03|11.2         |
|chicago    |1        |0     |chicago |2026-01-01 12:00:02|8.4          |
|denver     |2        |0     |denver  |2026-01-01 12:00:05|4.1          |
|boston     |2        |1     |boston  |2026-01-01 12:00:00|12.1         |
|denver     |2        |2     |denver  |2026-01-01 12:00:01|3.8          |
|boston     |2        |3     |boston  |2026-01-01 12:00:04|12.6         |
+-----------+---------+------+--------+-------------------+-------------+



In [8]:
# What forgetting the cast costs. Spark refuses the query outright rather than
# handing back a column of nulls, which is the kinder of the two failures.
try:
    raw.select(from_json(col("value"), PAYLOAD).alias("parsed")).show()
    print("no error: unexpected")
except AnalysisException as exc:
    print(str(exc).strip().split("\n")[0])

[DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve "from_json(value)" due to data type mismatch: The first parameter requires the "STRING" type, however "value" has the type "BINARY". SQLSTATE: 42K09;


## Keys decide partitions

Every reading from one station is in one partition, and a partition may hold more than
one station. That is the whole of Kafka's ordering guarantee: order is promised within a
partition, never across a topic.

In [9]:
(
    parsed.groupBy("partition")
    .agg({"station": "collect_set", "offset": "count"})
    .withColumnRenamed("collect_set(station)", "stations")
    .withColumnRenamed("count(offset)", "records")
    .orderBy("partition")
    .show(truncate=False)
)

+---------+-------+----------------+
|partition|records|stations        |
+---------+-------+----------------+
|0        |1      |[portland]      |
|1        |1      |[chicago]       |
|2        |4      |[denver, boston]|
+---------+-------+----------------+



## A topic is a log, so reads are repeatable

Nothing is consumed by being read. The same query run again returns the same rows, and a
range of offsets can be asked for by number. This is the property the chapter calls
*replayable*, and it is the first of the three conditions for end-to-end exactly-once.

In [10]:
again = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .load()
)
print("first read: ", raw.count(), "records")
print("second read:", again.count(), "records")
assert raw.count() == again.count()

first read:  6 records


second read: 6 records


In [11]:
# Just the first record of each partition, asked for by offset. Nothing was
# rewound and no state was reset; the log simply still has them.
first_only = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", json.dumps({TOPIC: {"0": 0, "1": 0, "2": 0}}))
    .option("endingOffsets", json.dumps({TOPIC: {"0": 1, "1": 1, "2": 1}}))
    .load()
)
first_only.selectExpr("partition", "offset", "CAST(key AS STRING) AS station").orderBy("partition").show(truncate=False)

+---------+------+--------+
|partition|offset|station |
+---------+------+--------+
|0        |0     |portland|
|1        |0     |chicago |
|2        |0     |denver  |
+---------+------+--------+



## The streaming form differs by one method name

`readStream` in place of `read`, and a trigger that says "process what is here, then
stop". Everything between is the same code. That is the chapter's central claim about
Structured Streaming, and here it is with nothing else changed.

No checkpoint is configured, because the sink here is Spark's in-memory table, which is
a debugging sink and cannot resume from one anyway. A real query would name a durable
checkpoint location, as `09.01 Kafka Weather Stream.py` does with `--checkpoint`.

In [12]:
streamed = (
    spark.readStream.format("kafka")          # read -> readStream, and nothing else
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .selectExpr("CAST(key AS STRING) AS station_key", "CAST(value AS STRING) AS payload")
    .select("station_key", from_json(col("payload"), PAYLOAD).alias("r"))
    .select("station_key", "r.*")
)

query = (
    streamed.writeStream.queryName("from_kafka")
    .outputMode("append")
    .format("memory")
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()
spark.table("from_kafka").orderBy("reading_time").show(truncate=False)

+-----------+--------+-------------------+-------------+
|station_key|station |reading_time       |temperature_c|
+-----------+--------+-------------------+-------------+
|boston     |boston  |2026-01-01 12:00:00|12.1         |
|denver     |denver  |2026-01-01 12:00:01|3.8          |
|chicago    |chicago |2026-01-01 12:00:02|8.4          |
|portland   |portland|2026-01-01 12:00:03|11.2         |
|boston     |boston  |2026-01-01 12:00:04|12.6         |
|denver     |denver  |2026-01-01 12:00:05|4.1          |
+-----------+--------+-------------------+-------------+



In [13]:
# The batch read and the streaming read returned the same records. An assertion,
# rather than a sentence claiming they agree.
batch_rows = parsed.select("station", "reading_time", "temperature_c").collect()
stream_rows = spark.table("from_kafka").select("station", "reading_time", "temperature_c").collect()
assert sorted(map(tuple, batch_rows)) == sorted(map(tuple, stream_rows))
print(len(batch_rows), "records, both ways, identical")

6 records, both ways, identical


## What this shows

A Kafka topic is a partitioned, durable log, and Spark reads it as a table of bytes with
a little metadata attached. Everything else follows from that. The cast and the declared
schema are needed because Kafka stores no types. The partition and offset pair names a
record, which is what lets a query stop and resume. Reading changes nothing, which is
what lets a failed batch be replayed rather than lost.

The one method name separating the batch read from the streaming read is not a
convenience. It is the claim the whole chapter rests on: the same query, the same
optimizer and the same tasks, run once over a bounded topic or repeatedly over an
unbounded one.

In [14]:
# Leave the broker as it was found.
admin.delete_topics([TOPIC])
admin.close()
spark.stop()
print("topic removed, session stopped")

topic removed, session stopped
